In [1]:
import torch
from pathlib import Path

from ultralytics import YOLO
import ultralytics
import ultralytics.nn.tasks as tasks

from ultralytics.nn.modules import CBAM as OriginalCBAM

print("Ultralytics:", ultralytics.__version__)
print("Original CBAM:", OriginalCBAM)

Ultralytics: 8.4.126
Original CBAM: <class 'ultralytics.nn.modules.conv.CBAM'>


In [2]:
# ============================================================
# PARSER-COMPATIBLE CBAM
# ============================================================

class EcoBotXCBAM(OriginalCBAM):

    def __init__(self, c1, kernel_size=7):
        super().__init__(
            c1=c1,
            kernel_size=kernel_size
        )


print("=" * 70)
print("EcoBotX CBAM")
print("=" * 70)

print(EcoBotXCBAM)

EcoBotX CBAM
<class '__main__.EcoBotXCBAM'>


In [3]:
# ============================================================
# REGISTER EcoBotXCBAM
# ============================================================

tasks.EcoBotXCBAM = EcoBotXCBAM

print("=" * 70)
print("REGISTERING EcoBotXCBAM")
print("=" * 70)

print("Registered class:")
print(tasks.EcoBotXCBAM)

print("\nConstructor:")
import inspect
print(inspect.signature(EcoBotXCBAM))

print("\n[OK] EcoBotXCBAM registered.")

REGISTERING EcoBotXCBAM
Registered class:
<class '__main__.EcoBotXCBAM'>

Constructor:
(c1, kernel_size=7)

[OK] EcoBotXCBAM registered.


In [4]:
# ============================================================
# EXPERIMENT 4 YAML
# ============================================================

PROJECT_DIR = Path(
    r"G:\EcoBotX_YOLO_training\experiment4"
)

PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_YAML = PROJECT_DIR / "ecobotx_net.yaml"


yaml_content = """
nc: 4

depth_multiple: 0.33
width_multiple: 0.25

backbone:

  # P1
  - [-1, 1, Conv, [64, 3, 2]]

  # P2 / 4
  - [-1, 1, Conv, [128, 3, 2]]

  # P2 feature
  - [-1, 3, C2f, [128, True]]

  # CBAM P2
  - [-1, 1, EcoBotXCBAM, [7]]

  # P3 / 8
  - [-1, 1, Conv, [256, 3, 2]]

  # P3 feature
  - [-1, 6, C2f, [256, True]]

  # CBAM P3
  - [-1, 1, EcoBotXCBAM, [7]]

  # P4 / 16
  - [-1, 1, Conv, [512, 3, 2]]

  # P4 feature
  - [-1, 6, C2f, [512, True]]

  # CBAM P4
  - [-1, 1, EcoBotXCBAM, [7]]

  # P5 / 32
  - [-1, 1, Conv, [1024, 3, 2]]

  # P5 feature
  - [-1, 3, C2f, [1024, True]]

  # SPPF
  - [-1, 1, SPPF, [1024, 5]]

  # CBAM P5
  - [-1, 1, EcoBotXCBAM, [7]]


head:

  # P5 -> P4
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]

  # P4 -> P3
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 7], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  # P3 -> P2
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]

  # P2 detection refinement
  - [-1, 1, EcoBotXCBAM, [7]]

  # P2 -> P3
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 17], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  # P3 -> P4
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 14], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]

  # P4 -> P5
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 11], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]

  # P2 + P3 + P4 + P5
  - [[21, 24, 27, 30], 1, Detect, [nc]]
"""


MODEL_YAML.write_text(
    yaml_content.strip(),
    encoding="utf-8"
)

print("=" * 70)
print("YAML CREATED")
print("=" * 70)

print(MODEL_YAML)

YAML CREATED
G:\EcoBotX_YOLO_training\experiment4\ecobotx_net.yaml


In [5]:
# ============================================================
# VERIFY YAML
# ============================================================

text = MODEL_YAML.read_text(
    encoding="utf-8"
)

print("=" * 70)
print("YAML VERIFICATION")
print("=" * 70)

print("EcoBotXCBAM occurrences:",
      text.count("EcoBotXCBAM"))

print("Old CBAM occurrences:",
      text.count("CBAM, [7]"))

if "CBAM, [7]" not in text:
    print("\n[OK] Incorrect CBAM syntax removed.")

if text.count("EcoBotXCBAM") == 5:
    print("[OK] Five CBAM modules configured.")
else:
    print("[WARNING] Expected five CBAM modules.")

YAML VERIFICATION
EcoBotXCBAM occurrences: 5
Old CBAM occurrences: 5
[OK] Five CBAM modules configured.


In [6]:
# ============================================================
# BUILD EXPERIMENT 4
# ============================================================

print("=" * 70)
print("BUILDING EXPERIMENT 4")
print("EcoBotX-Light + CBAM + P2")
print("=" * 70)

custom_p2_model = YOLO(
    str(MODEL_YAML)
)

print("\n[OK] Model created successfully.")

BUILDING EXPERIMENT 4
EcoBotX-Light + CBAM + P2


RuntimeError: Given groups=1, weight of size [7, 7, 1, 1], expected input[1, 32, 1, 1] to have 7 channels, but got 32 channels instead